# 🧠 Machine Learning from Scratch — Week 1 Capstone
**Author:** Walid Ayache  
**Internship:** Data Science Internship 2026  
**Focus:** Mathematical Foundations, Pure NumPy Vectorization, and Clean ML Architectures

---

## 📌 Overview
In this notebook, we implement core machine learning algorithms and evaluation metrics entirely from scratch using pure NumPy:
1. **Data Preprocessing:** `train_test_split` and `StandardScaler` (fit/transform paradigm)
2. **Linear Regression:** Normal Equation (closed-form) and Gradient Descent
3. **k-Nearest Neighbors (k-NN) Classifier:** Fully vectorized pairwise distance matrix and plurality voting
4. **Evaluation Metrics:** Accuracy, Precision, Recall, F1-Score, MSE, RMSE, MAE, and $R^2$
5. **Benchmarking & Validation:** Direct verification against `scikit-learn`

In [9]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# Scikit-Learn imports used strictly for benchmarking
from sklearn.linear_model import LinearRegression as SkLinearRegression
from sklearn.neighbors import KNeighborsClassifier as SkKNN
from sklearn import metrics as sk_metrics
from sklearn.datasets import make_classification, make_regression

print("NumPy & libraries loaded successfully!")

NumPy & libraries loaded successfully!


---  
## Part 1: Preprocessing & Data Utilities

### 1.1 `train_test_split`
Shuffles sample indices together and splits $X$ and $y$ into train and test sets.

In [10]:
def train_test_split(X, y, test_size=0.2, random_state=42):
    """Split arrays into random train and test subsets."""
    assert len(X) == len(y), "X and y must have identical sample length"
    n = len(X)
    rng = np.random.default_rng(random_state)
    shuffled_indices = rng.permutation(n)
    
    n_test = int(n * test_size)
    test_idx = shuffled_indices[:n_test]
    train_idx = shuffled_indices[n_test:]
    
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

### 1.2 `StandardScaler`
Standardizes features by removing the mean and scaling to unit variance:
$$z = \frac{x - \mu}{\sigma}$$
> **Rule:** Always fit on the training set only to prevent data leakage.

In [11]:
class StandardScaler:
    """Standardize features by centering mean to 0 and std to 1."""
    def __init__(self):
        self.mean_ = None
        self.std_ = None
        
    def fit(self, X):
        self.mean_ = np.mean(X, axis=0)
        self.std_ = np.std(X, axis=0)
        # Protect against division by zero
        self.std_[self.std_ == 0.0] = 1.0
        return self
        
    def transform(self, X):
        assert self.mean_ is not None, "Must call fit() before transform()"
        return (X - self.mean_) / self.std_
        
    def fit_transform(self, X):
        return self.fit(X).transform(X)

---  
## Part 2: Linear Regression from Scratch

### 2.1 Normal Equation (Closed-Form)
$$\mathbf{w} = (X_b^T X_b)^{-1} X_b^T y$$

In [12]:
class LinearRegressionNormal:
    def __init__(self):
        self.weights_ = None
        
    def fit(self, X, y):
        n = X.shape[0]
        X_b = np.column_stack([X, np.ones(n)])
        self.weights_ = np.linalg.pinv(X_b.T @ X_b) @ X_b.T @ y
        return self
        
    def predict(self, X):
        n = X.shape[0]
        X_b = np.column_stack([X, np.ones(n)])
        return X_b @ self.weights_

# Verify with synthetic regression data
X_reg, y_reg = make_regression(n_samples=100, n_features=2, noise=10.0, random_state=42)
X_r_train, X_r_test, y_r_train, y_r_test = train_test_split(X_reg, y_reg, test_size=0.2)

my_lr = LinearRegressionNormal().fit(X_r_train, y_r_train)
sk_lr = SkLinearRegression().fit(X_r_train, y_r_train)

preds_my = my_lr.predict(X_r_test)
preds_sk = sk_lr.predict(X_r_test)

assert np.allclose(preds_my, preds_sk), "Linear Regression Normal Eq differs from sklearn!"
print("✅ Linear Regression (Normal Equation) matches sklearn exactly!")

✅ Linear Regression (Normal Equation) matches sklearn exactly!


---  
## Part 3: k-Nearest Neighbors (k-NN) Classifier from Scratch

### Pairwise Distance Matrix (Vectorized Identity)
$$\|\mathbf{a} - \mathbf{b}\|^2 = \|\mathbf{a}\|^2 + \|\mathbf{b}\|^2 - 2 \mathbf{a}^T \mathbf{b}$$

In [13]:
class KNNClassifier:
    def __init__(self, k=3, metric='euclidean'):
        self.k = int(k)
        self.metric = metric
        self.X_train = None
        self.y_train = None
        self.classes_ = None
        
    def fit(self, X, y):
        self.X_train = np.asarray(X, dtype=float).copy()
        self.y_train = np.asarray(y).copy()
        self.classes_ = np.unique(y)
        return self
        
    def _compute_dists(self, X_test):
        if self.metric == 'euclidean':
            test_sq = np.sum(X_test ** 2, axis=1, keepdims=True)
            train_sq = np.sum(self.X_train ** 2, axis=1, keepdims=True).T
            sq_dists = test_sq + train_sq - 2.0 * (X_test @ self.X_train.T)
            return np.sqrt(np.maximum(sq_dists, 0.0))
        else:
            return np.sum(np.abs(X_test[:, np.newaxis, :] - self.X_train[np.newaxis, :, :]), axis=2)
            
    def predict(self, X):
        X = np.asarray(X, dtype=float)
        dists = self._compute_dists(X)
        k_indices = np.argsort(dists, axis=1)[:, :self.k]
        k_labels = self.y_train[k_indices]
        
        preds = [Counter(row).most_common(1)[0][0] for row in k_labels]
        return np.array(preds)
        
    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        dists = self._compute_dists(X)
        k_indices = np.argsort(dists, axis=1)[:, :self.k]
        k_labels = self.y_train[k_indices]
        
        probas = np.zeros((len(X), len(self.classes_)))
        for i, row in enumerate(k_labels):
            for j, c in enumerate(self.classes_):
                probas[i, j] = np.mean(row == c)
        return probas

---  
## Part 4: Evaluation Metrics Suite from Scratch

### 4.1 Classification Metrics: Accuracy, Precision, Recall, F1, Confusion Matrix

In [14]:
def confusion_matrix(y_true, y_pred, labels=None):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    if labels is None:
        labels = np.unique(np.concatenate([y_true, y_pred]))
    label_to_idx = {val: i for i, val in enumerate(labels)}
    cm = np.zeros((len(labels), len(labels)), dtype=int)
    for yt, yp in zip(y_true, y_pred):
        if yt in label_to_idx and yp in label_to_idx:
            cm[label_to_idx[yt], label_to_idx[yp]] += 1
    return cm

def accuracy_score(y_true, y_pred):
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))

def precision_score(y_true, y_pred, pos_label=1):
    yt, yp = np.asarray(y_true), np.asarray(y_pred)
    tp = np.sum((yt == pos_label) & (yp == pos_label))
    fp = np.sum((yt != pos_label) & (yp == pos_label))
    return float(tp / (tp + fp)) if (tp + fp) > 0 else 0.0

def recall_score(y_true, y_pred, pos_label=1):
    yt, yp = np.asarray(y_true), np.asarray(y_pred)
    tp = np.sum((yt == pos_label) & (yp == pos_label))
    fn = np.sum((yt == pos_label) & (yp != pos_label))
    return float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0

def f1_score(y_true, y_pred, pos_label=1):
    p = precision_score(y_true, y_pred, pos_label)
    r = recall_score(y_true, y_pred, pos_label)
    return float(2 * (p * r) / (p + r)) if (p + r) > 0 else 0.0

### 4.2 Regression Metrics: MSE, RMSE, MAE, $R^2$

In [15]:
def mean_squared_error(y_true, y_pred):
    return float(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))

def root_mean_squared_error(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mean_absolute_error(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))

def r2_score(y_true, y_pred):
    yt, yp = np.asarray(y_true), np.asarray(y_pred)
    ss_res = np.sum((yt - yp) ** 2)
    ss_tot = np.sum((yt - np.mean(yt)) ** 2)
    return float(1.0 - (ss_res / ss_tot)) if ss_tot != 0 else 0.0

---  
## Part 5: End-to-End Pipeline & Scikit-Learn Benchmarking

In [ ]:
# Generate classification benchmark data
X_clf, y_clf = make_classification(n_samples=300, n_features=4, n_informative=3,n_redundant=1, n_classes=2, random_state=42)
X_c_train, X_c_test, y_c_train, y_c_test = train_test_split(X_clf, y_clf, test_size=0.25, random_state=42)

# Preprocess with StandardScaler
scaler = StandardScaler()
X_c_train_scaled = scaler.fit_transform(X_c_train)
X_c_test_scaled = scaler.transform(X_c_test)

# Fit custom KNN
my_knn = KNNClassifier(k=5, metric='euclidean').fit(X_c_train_scaled, y_c_train)
sk_knn = SkKNN(n_neighbors=5, metric='euclidean', algorithm='brute').fit(X_c_train_scaled, y_c_train)

my_preds = my_knn.predict(X_c_test_scaled)
sk_preds = sk_knn.predict(X_c_test_scaled)

# Verify accuracy and f1 against sklearn
assert np.array_equal(my_preds, sk_preds), "k-NN predictions differ from sklearn!"
assert np.isclose(accuracy_score(y_c_test, my_preds), sk_metrics.accuracy_score(y_c_test, sk_preds))
assert np.isclose(f1_score(y_c_test, my_preds), sk_metrics.f1_score(y_c_test, sk_preds))

print(f"Custom k-NN Accuracy: {accuracy_score(y_c_test, my_preds):.4f}")
print(f"Custom k-NN F1-Score: {f1_score(y_c_test, my_preds):.4f}")
print("✅ Custom k-NN and Metrics suite match scikit-learn 100%!")

ValueError: Number of informative, redundant and repeated features must sum to less than the number of total features

---  
## 📈 Decision Boundary Visualization

In [ ]:
# Train on 2 features for 2D visualization
X_2d = X_c_train_scaled[:, :2]
knn_2d = KNNClassifier(k=5).fit(X_2d, y_c_train)

x_min, x_max = X_2d[:, 0].min() - 0.5, X_2d[:, 0].max() + 0.5
y_min, y_max = X_2d[:, 1].min() - 0.5, X_2d[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 150), np.linspace(y_min, y_max, 150))
Z = knn_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.coolwarm)
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y_c_train, cmap=plt.cm.coolwarm, edgecolors='k')
plt.title("k-NN (k=5) Decision Boundary (from Scratch)")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()